# avg


In [1]:

#com valores 60 e 120 hard coded

import random
import numpy as np
import math
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt  # Importar matplotlib para plotar gráficos
from functools import partial  # Adicionar no início do código
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

NUM_UAVS = 4
lambda_0 = 0.125
P_INTERFERENCE_DBM = 100
P_NOISE_DBM = -100
BANDWIDTH = 20e6
TRANSMIT_POWER_DBM = 20
MINDIST = 20
RESOLUTION = 20

# Definir os tipos de indivíduos e fitness
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximizar o fitnessAS
creator.create("Individual", list, fitness=creator.FitnessMax)  # Indivíduo é uma listaas

#ATENCAO JAMMER POSITION NO CREATE INDIVIDUAL

# Nova função para criar um indivíduo
def create_individual(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, epoch_total_length=200, previous_best_solution=None, initial_positions=None, jammer_position=None):
    individual = []
    
    # Definir as posições iniciais de acordo com o epoch
    if epoch == 0:
        start_positions = initial_positions
    else:
        start_positions = []
        for uav in range(num_uavs):
            last_timeslot_idx = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            start_positions.append(previous_best_solution[last_timeslot_idx:last_timeslot_idx + 2])
    
    # Gerar posições finais CONTÍNUAS respeitando MINDIST
    final_positions = []
    max_attempts = 1000  # Evitar loop infinito
    
    for uav in range(num_uavs):
        attempts = 0
        while attempts < max_attempts:
            # Gerar posição final aleatória contínua
            final_x = random.uniform(60, 120)
            final_y = random.uniform(0, 60)
            candidate_position = (final_x, final_y)
            
            # Verificar se respeita MINDIST com todas as posições já geradas
            valid = True
            for existing_pos in final_positions:
                distance = np.linalg.norm(np.array(candidate_position) - np.array(existing_pos))
                if distance < MINDIST:
                    valid = False
                    break
            
            if valid:
                final_positions.append(candidate_position)
                break
            
            attempts += 1
        
        # Se não conseguir após max_attempts, aceita a posição mesmo assim
        if attempts >= max_attempts:
            final_x = random.uniform(60, 120)
            final_y = random.uniform(0, 60)
            final_positions.append((final_x, final_y))
    
    # Calcular todas as posições intermediárias
    for t in range(num_timeslots):
        for uav in range(num_uavs):
            start_x, start_y = start_positions[uav]
            end_x, end_y = final_positions[uav]
            
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            x = start_x + alpha * (end_x - start_x)
            y = start_y + alpha * (end_y - start_y)
            
            individual.extend([x, y])
    
    return creator.Individual(individual)

def print_communication_values_per_timeslot(individual, num_uavs, num_timeslots, jammer_position):
    all_min_capacities = []
    all_interference_matrices = []
    all_fitness_values = []  # ← ADICIONAR para armazenar fitness de cada timeslot

    for t in range(num_timeslots):
        angles = []
        positions = []
        
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        comm_matrix, interference_matrix = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)
        all_interference_matrices.append(interference_matrix)

        # ← USAR A MESMA LÓGICA DO evaluate_individual
        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))
                    except:
                        pass
        
        # Cálculo usando apenas links usados (IGUAL AO evaluate_individual)
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
            all_min_capacities.extend(capacidades_usadas)  # ← Adicionar todas as capacidades usadas
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Calcular fitness do timeslot (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        all_fitness_values.append(fitness)

    # Calcular fitness médio (IGUAL AO evaluate_individual)
    if len(all_fitness_values) > 0:
        avg_fitness = sum(all_fitness_values) / len(all_fitness_values)
    else:
        avg_fitness = 0.0

    if len(all_min_capacities) > 0:
        avg_min_capacity = sum(all_min_capacities) / len(all_min_capacities)
    else:
        avg_min_capacity = 0.0

    return all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness  # ← ADICIONAR avg_fitness

# Função para calcular a capacidade de comunicação entre todos os UAVs
def calculate_communication_capacity(antenna_angles, alignments, positions, jammer_position):
    communication_capacity = np.zeros((NUM_UAVS, NUM_UAVS))
    interference_matrix = np.full((NUM_UAVS, NUM_UAVS), P_INTERFERENCE_DBM)

    for i in range(NUM_UAVS):
        null_dir_i = antenna_angles[i]

        dx_jam = jammer_position[0] - positions[i][0]
        dy_jam = jammer_position[1] - positions[i][1]
        dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

        G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

        dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
        P_interf_dBm_i = interference_from_jammer(P_INTERFERENCE_DBM, G_jammer_i, lambda_0, dist_jammer_i)

        interference_matrix[i, :] = P_interf_dBm_i

    for i in range(NUM_UAVS):
        for j in range(NUM_UAVS):
            if i != j:
                null_dir_i = antenna_angles[i]
                null_dir_j = antenna_angles[j]

                dx_ij = positions[j][0] - positions[i][0]
                dy_ij = positions[j][1] - positions[i][1]
                dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                dx_ji = positions[i][0] - positions[j][0]
                dy_ji = positions[i][1] - positions[j][1]
                dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                capacity = calcular_capacidade_link(
                    pos1=positions[i],
                    pos2=positions[j],
                    lambda_0=lambda_0,
                    P_tx_dBm=TRANSMIT_POWER_DBM,
                    G_tx_dB=G_tx,
                    G_rx_dB=G_rx,
                    P_interference_dBm=interference_matrix[i, j],
                    P_noise_dBm=P_NOISE_DBM,
                    bandwidth=BANDWIDTH
                )

                communication_capacity[i][j] = capacity

    return communication_capacity, interference_matrix

def evaluate_individual(individual, num_uavs, num_timeslots, jammer_position):
    # Verificar se há pelo menos uma colisão
    
    # Se não há colisões, calcular o fitness normalmente
    alpha, beta = 1.0, 1.0
    total_fitness = 0

    for t in range(num_timeslots):
        positions = []
        angles = []
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)

        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Fitness sem penalização por colisões (já sabemos que não há colisões)
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        total_fitness += fitness

    # Calcular a média do fitness total
    average_fitness = total_fitness / num_timeslots if num_timeslots > 0 else 0
    return (average_fitness,)

def mutate_individual(individual, min_x, max_x, min_y, max_y, mutation_rate, num_uavs, num_timeslots, epoch, timeslot_length, epoch_total_length):
    
    for uav in range(num_uavs):
        if random.random() < mutation_rate:
            # Extrair posições iniciais
            start_x = individual[uav * 2]
            start_y = individual[uav * 2 + 1]
            
            # Obter posições finais atuais de todos os outros UAVs
            other_final_positions = []
            for other_uav in range(num_uavs):
                if other_uav != uav:
                    final_x = individual[(num_timeslots - 1) * num_uavs * 2 + other_uav * 2]
                    final_y = individual[(num_timeslots - 1) * num_uavs * 2 + other_uav * 2 + 1]
                    other_final_positions.append((final_x, final_y))
            
            # Tentar gerar nova posição final respeitando MINDIST
            max_attempts = 1000
            attempts = 0
            new_final_x, new_final_y = None, None
            
            while attempts < max_attempts:
                candidate_x = random.uniform(60, 120)
                candidate_y = random.uniform(0, 60)
                candidate_position = (candidate_x, candidate_y)
                
                # Verificar se respeita MINDIST com todas as outras posições finais
                valid = True
                for other_pos in other_final_positions:
                    distance = np.linalg.norm(np.array(candidate_position) - np.array(other_pos))
                    if distance < MINDIST:
                        valid = False
                        break
                
                if valid:
                    new_final_x, new_final_y = candidate_x, candidate_y
                    break
                
                attempts += 1
            
            # Se não conseguir, manter a posição atual (não fazer mutação)
            if new_final_x is None:
                continue
            
            # Atualizar a posição final
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2] = new_final_x
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1] = new_final_y
            
            # Recalcular posições intermediárias
            for t in range(1, num_timeslots):
                alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                x = start_x + alpha * (new_final_x - start_x)
                y = start_y + alpha * (new_final_y - start_y)
                
                idx_x = t * num_uavs * 2 + uav * 2
                idx_y = t * num_uavs * 2 + uav * 2 + 1
                individual[idx_x] = x
                individual[idx_y] = y
    
    return individual,

def custom_crossover(ind1, ind2, num_uavs):
    num_timeslots = len(ind1) // (num_uavs * 2)
    
    # Função auxiliar para verificar se posições finais respeitam MINDIST
    def check_mindist_final_positions(individual):
        final_positions = []
        for uav in range(num_uavs):
            final_idx_x = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            final_idx_y = (num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1
            final_positions.append((individual[final_idx_x], individual[final_idx_y]))
        
        # Verificar todas as combinações de pares
        for i in range(len(final_positions)):
            for j in range(i + 1, len(final_positions)):
                distance = np.linalg.norm(np.array(final_positions[i]) - np.array(final_positions[j]))
                if distance < MINDIST:
                    return False
        return True
    
    # Fazer cópias dos indivíduos originais
    original_ind1 = ind1[:]
    original_ind2 = ind2[:]
    
    # Trocar posições finais entre os indivíduos
    for uav in range(num_uavs):
        if random.random() < 0.8:
            # Índices das posições finais (último timeslot)
            final_idx_x = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            final_idx_y = (num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1
            
            # Trocar as posições finais
            ind1[final_idx_x], ind2[final_idx_x] = ind2[final_idx_x], ind1[final_idx_x]
            ind1[final_idx_y], ind2[final_idx_y] = ind2[final_idx_y], ind1[final_idx_y]
            
            # Verificar se ambos os indivíduos ainda respeitam MINDIST
            if not (check_mindist_final_positions(ind1) and check_mindist_final_positions(ind2)):
                # Se não respeitam, reverter a troca
                ind1[final_idx_x], ind2[final_idx_x] = ind2[final_idx_x], ind1[final_idx_x]
                ind1[final_idx_y], ind2[final_idx_y] = ind2[final_idx_y], ind1[final_idx_y]
                continue
            
            # Se respeitam MINDIST, recalcular posições intermediárias
            for ind in [ind1, ind2]:
                start_x = ind[uav * 2]
                start_y = ind[uav * 2 + 1]
                end_x = ind[final_idx_x]
                end_y = ind[final_idx_y]
                
                for t in range(1, num_timeslots - 1):
                    alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                    idx_x = t * num_uavs * 2 + uav * 2
                    idx_y = t * num_uavs * 2 + uav * 2 + 1
                    ind[idx_x] = start_x + alpha * (end_x - start_x)
                    ind[idx_y] = start_y + alpha * (end_y - start_y)
    
    return ind1, ind2

# Configuração do DEAP atualizada
def setup_deap(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, crossover_rate, mutation_rate, initial_positions, previous_best_solution=None, jammer_position=None, epoch_total_length=300):
    toolbox = base.Toolbox()
    
    # Registrar funções
    toolbox.register("individual", create_individual, 
                     num_uavs=num_uavs, 
                     num_timeslots=num_timeslots, 
                     timeslot_length=timeslot_length, 
                     epoch=epoch, 
                     min_y=min_y, 
                     max_y=max_y,
                     epoch_total_length=epoch_total_length,  # ← ADICIONAR
                     previous_best_solution=previous_best_solution,
                     initial_positions=initial_positions,
                     jammer_position=jammer_position)  # Passar a posição do jammer aqui
    
    # O restante do código permanece o mesmo...

    
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate_individual, num_uavs=num_uavs, num_timeslots=num_timeslots, jammer_position=jammer_position)
    
    # Usar partial para fixar o argumento num_uavs na função custom_crossover
    toolbox.register("mate", partial(custom_crossover, num_uavs=num_uavs))
    
    min_x = epoch * num_timeslots * timeslot_length
    max_x = (epoch * num_timeslots + num_timeslots) * timeslot_length
    
    toolbox.register("mutate", mutate_individual, 
                 min_x=min_x, 
                 max_x=max_x, 
                 min_y=min_y, 
                 max_y=max_y, 
                 mutation_rate=mutation_rate,
                 num_uavs=num_uavs,
                 num_timeslots=num_timeslots,
                 epoch=epoch,
                 timeslot_length=timeslot_length,
                 epoch_total_length=epoch_total_length)  # ← ADICIONAR
    
    toolbox.register("select", tools.selTournament, tournsize=3)  # Seleção por torneio
    
    return toolbox

def genetic_algorithm(num_uavs, num_timeslots, timeslot_length, population_size, generations, crossover_rate, mutation_rate, epoch, initial_positions, previous_best_solution=None, jammer_position=None, min_y=0.0, max_y=5.0, epoch_total_length=300):
    # Configurar o DEAP
    toolbox = setup_deap(
        num_uavs=num_uavs,
        num_timeslots=num_timeslots,
        timeslot_length=timeslot_length,
        epoch=epoch,
        min_y=min_y,
        max_y=max_y,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        initial_positions=initial_positions,
        previous_best_solution=previous_best_solution,
        jammer_position=jammer_position,
        epoch_total_length=epoch_total_length  # ← ADICIONAR
    )

    
    # Criar população inicial
    population = toolbox.population(n=population_size)
    
    # Avaliar a população inicial
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
    
    # Configurar estatísticas para impressão
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Listas para armazenar os dados de cada geração
    gen_list = []
    avg_list = []
    std_list = []
    min_list = []
    max_list = []
    
    # Executar o algoritmo genético com eaSimple
    for gen in range(generations):
        # Avançar uma geração
        algorithms.eaSimple(
            population, 
            toolbox, 
            cxpb=crossover_rate,  # Probabilidade de cruzamento
            mutpb=mutation_rate,  # Probabilidade de mutação
            ngen=1,               # Apenas uma geração por iteração
            stats=stats,          # Estatísticas para impressão
            verbose=False         # Desativar impressão da tabela para cada geração
        )
        
        # Coletar os dados da geração atual
        record = stats.compile(population)
        gen_list.append(gen)
        avg_list.append(record["avg"])
        std_list.append(record["std"])
        min_list.append(record["min"])
        max_list.append(record["max"])
        
        # Escrever os valores no arquivo
        write_fitness_values(epoch, gen, record["avg"], record["max"], record["min"], record["std"])
    
    # Retornar o melhor indivíduo
    best_individual = tools.selBest(population, k=1)[0]
    return best_individual

# Função para gerar posições iniciais dos UAVs
def generate_initial_positions(num_uavs, min_y, max_y, timeslot_length, manual=False, manual_positions=None):
    if manual and manual_positions is not None:
        return manual_positions  # Usa as posições fornecidas

    positions = []
    max_attempts = 1000  # para evitar loops infinitos

    for _ in range(num_uavs):
        attempts = 0
        while True:
            y = random.uniform(min_y, max_y)
            x = random.uniform(0, timeslot_length)  # entre -timeslot_length e 0
            
            candidate = (x, y)
            
            # Verifica se está longe o suficiente das outras posições já geradas
            if all(np.linalg.norm(np.array(candidate) - np.array(pos)) >= MINDIST for pos in positions):
                positions.append(candidate)
                break
            
            attempts += 1
            if attempts >= max_attempts:
                # Se não conseguir, aceita a posição mesmo assim para evitar bloqueio
                positions.append(candidate)
                break
                
    return positions

# def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
#     with open(filename, "a") as file:
#         # Escrever apenas a melhor solução (que já inclui tudo)
#         individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
#         file.write(f"{individual_str}\n \n")

def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            # Processar o conteúdo existente se necessário
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        # Se o arquivo não existir, não há conteúdo para ler
        print("O arquivo não existe. Criando um novo arquivo.")

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")


def write_fitness_values(epoch, gen, avg, max_val, min_val, std, filename="fitness_values.txt"):
    
    with open(filename, "a") as file:
        if gen == 0:  # Escrever o cabeçalho no início de cada epoch
            file.write(f"=== Epoch {epoch + 1} ===\n")
            file.write("gen\tavg\tmax\tmin\tstd\n")
        file.write(f"{gen}\t{avg:.4f}\t{max_val:.4f}\t{min_val:.4f}\t{std:.4f}\n")

# Função principal atualizada
def simulate_uavs_with_ga(num_epochs, num_timeslots, timeslot_length, num_uavs, population_size=50, generations=50, crossover_rate=0.9, mutation_rate=0.3, manual_initial_positions=None, jammer_position=None, min_y=0.0, max_y=10.0, epoch_total_length=300):
    # Gerar posições iniciais dos UAVs
    initial_positions = generate_initial_positions(
        num_uavs, min_y, max_y, timeslot_length,
        manual=manual_initial_positions is not None,
        manual_positions=manual_initial_positions
    )

    previous_best_solution = None
    
    for epoch in range(num_epochs):
        # Executar o algoritmo genético
        best_solution = genetic_algorithm(
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            timeslot_length=timeslot_length,
            population_size=population_size,
            generations=generations,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            epoch=epoch,
            initial_positions=initial_positions,
            previous_best_solution=previous_best_solution,
            jammer_position=jammer_position,  # Passar a posição do jammer aqui
            min_y=min_y,
            max_y=max_y,
            epoch_total_length=epoch_total_length  # ← ADICIONAR
        )

        # O restante do código permanece o mesmo...


        # Salvar a melhor solução em um arquivo de texto
        save_best_solution_to_file(best_solution, initial_positions)
        
        all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness = print_communication_values_per_timeslot(best_solution, num_uavs, num_timeslots, jammer_position)

        # Guardar a melhor solução para a próxima epoch


        # # Calcular e imprimir a coerência dos valores
        # best_fitness = best_solution.fitness.values[0]
        # recalculated_fitness = evaluate_individual(best_solution, num_uavs, num_timeslots, jammer_position)[0]
        # # Ler o último valor máximo do fitness_values.txt
        # with open("fitness_values.txt", "r") as file:
        #     lines = file.readlines()
        #     last_max_fitness = float(lines[-1].split("\t")[2])  # Último max na última linha
        # # Comparar os valores
        # print(f"Fitness da melhor solução: {best_fitness:.4f}")
        # print(f"Fitness recalculado: {recalculated_fitness:.4f}")
        # print(f"Último max no arquivo: {last_max_fitness:.4f}")
        # print(f"São iguais? {'✅' if abs(best_fitness - last_max_fitness) < 1 else '❌'}")
        # # Guardar a melhor solução para a próxima epoch
        # previous_best_solution = best_solution



        previous_best_solution = best_solution

        return best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity


# prod


In [ ]:

#com valores 60 e 120 hard coded

import random
import numpy as np
import math
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt  # Importar matplotlib para plotar gráficos
from functools import partial  # Adicionar no início do código
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

NUM_UAVS = 4
lambda_0 = 0.125
P_INTERFERENCE_DBM = 100
P_NOISE_DBM = -100
BANDWIDTH = 20e6
TRANSMIT_POWER_DBM = 20
MINDIST = 20
RESOLUTION = 20

# Definir os tipos de indivíduos e fitness
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximizar o fitnessAS
creator.create("Individual", list, fitness=creator.FitnessMax)  # Indivíduo é uma listaas

#ATENCAO JAMMER POSITION NO CREATE INDIVIDUAL

# Nova função para criar um indivíduo
def create_individual(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, epoch_total_length=200, previous_best_solution=None, initial_positions=None, jammer_position=None):
    individual = []
    
    # Definir as posições iniciais de acordo com o epoch
    if epoch == 0:
        start_positions = initial_positions
    else:
        start_positions = []
        for uav in range(num_uavs):
            last_timeslot_idx = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            start_positions.append(previous_best_solution[last_timeslot_idx:last_timeslot_idx + 2])
    
    # Gerar posições finais CONTÍNUAS respeitando MINDIST
    final_positions = []
    max_attempts = 1000  # Evitar loop infinito
    
    for uav in range(num_uavs):
        attempts = 0
        while attempts < max_attempts:
            # Gerar posição final aleatória contínua
            final_x = random.uniform(60, 120)
            final_y = random.uniform(0, 60)
            candidate_position = (final_x, final_y)
            
            # Verificar se respeita MINDIST com todas as posições já geradas
            valid = True
            for existing_pos in final_positions:
                distance = np.linalg.norm(np.array(candidate_position) - np.array(existing_pos))
                if distance < MINDIST:
                    valid = False
                    break
            
            if valid:
                final_positions.append(candidate_position)
                break
            
            attempts += 1
        
        # Se não conseguir após max_attempts, aceita a posição mesmo assim
        if attempts >= max_attempts:
            final_x = random.uniform(60, 120)
            final_y = random.uniform(0, 60)
            final_positions.append((final_x, final_y))
    
    # Calcular todas as posições intermediárias
    for t in range(num_timeslots):
        for uav in range(num_uavs):
            start_x, start_y = start_positions[uav]
            end_x, end_y = final_positions[uav]
            
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            x = start_x + alpha * (end_x - start_x)
            y = start_y + alpha * (end_y - start_y)
            
            individual.extend([x, y])
    
    return creator.Individual(individual)

def print_communication_values_per_timeslot(individual, num_uavs, num_timeslots, jammer_position):
    all_min_capacities = []
    all_interference_matrices = []
    all_fitness_values = []  # ← ADICIONAR para armazenar fitness de cada timeslot

    for t in range(num_timeslots):
        angles = []
        positions = []
        
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        comm_matrix, interference_matrix = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)
        all_interference_matrices.append(interference_matrix)

        # ← USAR A MESMA LÓGICA DO evaluate_individual
        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))
                    except:
                        pass
        
        # Cálculo usando apenas links usados (IGUAL AO evaluate_individual)
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
            all_min_capacities.extend(capacidades_usadas)  # ← Adicionar todas as capacidades usadas
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Calcular fitness do timeslot (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        all_fitness_values.append(fitness)

    # Calcular fitness médio (IGUAL AO evaluate_individual)
    if len(all_fitness_values) > 0:
        avg_fitness = sum(all_fitness_values) / len(all_fitness_values)
    else:
        avg_fitness = 0.0

    if len(all_min_capacities) > 0:
        avg_min_capacity = sum(all_min_capacities) / len(all_min_capacities)
    else:
        avg_min_capacity = 0.0

    return all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness  # ← ADICIONAR avg_fitness

# Função para calcular a capacidade de comunicação entre todos os UAVs
def calculate_communication_capacity(antenna_angles, alignments, positions, jammer_position):
    communication_capacity = np.zeros((NUM_UAVS, NUM_UAVS))
    interference_matrix = np.full((NUM_UAVS, NUM_UAVS), P_INTERFERENCE_DBM)

    for i in range(NUM_UAVS):
        null_dir_i = antenna_angles[i]

        dx_jam = jammer_position[0] - positions[i][0]
        dy_jam = jammer_position[1] - positions[i][1]
        dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

        G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

        dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
        P_interf_dBm_i = interference_from_jammer(P_INTERFERENCE_DBM, G_jammer_i, lambda_0, dist_jammer_i)

        interference_matrix[i, :] = P_interf_dBm_i

    for i in range(NUM_UAVS):
        for j in range(NUM_UAVS):
            if i != j:
                null_dir_i = antenna_angles[i]
                null_dir_j = antenna_angles[j]

                dx_ij = positions[j][0] - positions[i][0]
                dy_ij = positions[j][1] - positions[i][1]
                dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                dx_ji = positions[i][0] - positions[j][0]
                dy_ji = positions[i][1] - positions[j][1]
                dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                capacity = calcular_capacidade_link(
                    pos1=positions[i],
                    pos2=positions[j],
                    lambda_0=lambda_0,
                    P_tx_dBm=TRANSMIT_POWER_DBM,
                    G_tx_dB=G_tx,
                    G_rx_dB=G_rx,
                    P_interference_dBm=interference_matrix[i, j],
                    P_noise_dBm=P_NOISE_DBM,
                    bandwidth=BANDWIDTH
                )

                communication_capacity[i][j] = capacity

    return communication_capacity, interference_matrix

def evaluate_individual(individual, num_uavs, num_timeslots, jammer_position):
    # Verificar se há pelo menos uma colisão
    #if has_collision(individual, num_uavs, num_timeslots):
        #return (0.0,)  # FITNESS 0 imediatamente
    
    # Se não há colisões, calcular o fitness normalmente
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []

    for t in range(num_timeslots):
        positions = []
        angles = []
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)

        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Fitness para este timeslot
        fitness_timeslot = (C_media_total ** alpha) * (C_min_total ** beta)
        fitness_por_timeslot.append(fitness_timeslot)

    # Calcular o fitness total como o produto dos fitness de cada timeslot
    total_fitness = np.prod(fitness_por_timeslot) if fitness_por_timeslot else 0
    return (total_fitness,)

def mutate_individual(individual, min_x, max_x, min_y, max_y, mutation_rate, num_uavs, num_timeslots, epoch, timeslot_length, epoch_total_length):
    
    for uav in range(num_uavs):
        if random.random() < mutation_rate:
            # Extrair posições iniciais
            start_x = individual[uav * 2]
            start_y = individual[uav * 2 + 1]
            
            # Obter posições finais atuais de todos os outros UAVs
            other_final_positions = []
            for other_uav in range(num_uavs):
                if other_uav != uav:
                    final_x = individual[(num_timeslots - 1) * num_uavs * 2 + other_uav * 2]
                    final_y = individual[(num_timeslots - 1) * num_uavs * 2 + other_uav * 2 + 1]
                    other_final_positions.append((final_x, final_y))
            
            # Tentar gerar nova posição final respeitando MINDIST
            max_attempts = 1000
            attempts = 0
            new_final_x, new_final_y = None, None
            
            while attempts < max_attempts:
                candidate_x = random.uniform(60, 120)
                candidate_y = random.uniform(0, 60)
                candidate_position = (candidate_x, candidate_y)
                
                # Verificar se respeita MINDIST com todas as outras posições finais
                valid = True
                for other_pos in other_final_positions:
                    distance = np.linalg.norm(np.array(candidate_position) - np.array(other_pos))
                    if distance < MINDIST:
                        valid = False
                        break
                
                if valid:
                    new_final_x, new_final_y = candidate_x, candidate_y
                    break
                
                attempts += 1
            
            # Se não conseguir, manter a posição atual (não fazer mutação)
            if new_final_x is None:
                continue
            
            # Atualizar a posição final
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2] = new_final_x
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1] = new_final_y
            
            # Recalcular posições intermediárias
            for t in range(1, num_timeslots):
                alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                x = start_x + alpha * (new_final_x - start_x)
                y = start_y + alpha * (new_final_y - start_y)
                
                idx_x = t * num_uavs * 2 + uav * 2
                idx_y = t * num_uavs * 2 + uav * 2 + 1
                individual[idx_x] = x
                individual[idx_y] = y
    
    return individual,

def custom_crossover(ind1, ind2, num_uavs):
    num_timeslots = len(ind1) // (num_uavs * 2)
    
    # Função auxiliar para verificar se posições finais respeitam MINDIST
    def check_mindist_final_positions(individual):
        final_positions = []
        for uav in range(num_uavs):
            final_idx_x = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            final_idx_y = (num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1
            final_positions.append((individual[final_idx_x], individual[final_idx_y]))
        
        # Verificar todas as combinações de pares
        for i in range(len(final_positions)):
            for j in range(i + 1, len(final_positions)):
                distance = np.linalg.norm(np.array(final_positions[i]) - np.array(final_positions[j]))
                if distance < MINDIST:
                    return False
        return True
    
    # Fazer cópias dos indivíduos originais
    original_ind1 = ind1[:]
    original_ind2 = ind2[:]
    
    # Trocar posições finais entre os indivíduos
    for uav in range(num_uavs):
        if random.random() < 0.8:
            # Índices das posições finais (último timeslot)
            final_idx_x = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            final_idx_y = (num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1
            
            # Trocar as posições finais
            ind1[final_idx_x], ind2[final_idx_x] = ind2[final_idx_x], ind1[final_idx_x]
            ind1[final_idx_y], ind2[final_idx_y] = ind2[final_idx_y], ind1[final_idx_y]
            
            # Verificar se ambos os indivíduos ainda respeitam MINDIST
            if not (check_mindist_final_positions(ind1) and check_mindist_final_positions(ind2)):
                # Se não respeitam, reverter a troca
                ind1[final_idx_x], ind2[final_idx_x] = ind2[final_idx_x], ind1[final_idx_x]
                ind1[final_idx_y], ind2[final_idx_y] = ind2[final_idx_y], ind1[final_idx_y]
                continue
            
            # Se respeitam MINDIST, recalcular posições intermediárias
            for ind in [ind1, ind2]:
                start_x = ind[uav * 2]
                start_y = ind[uav * 2 + 1]
                end_x = ind[final_idx_x]
                end_y = ind[final_idx_y]
                
                for t in range(1, num_timeslots - 1):
                    alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                    idx_x = t * num_uavs * 2 + uav * 2
                    idx_y = t * num_uavs * 2 + uav * 2 + 1
                    ind[idx_x] = start_x + alpha * (end_x - start_x)
                    ind[idx_y] = start_y + alpha * (end_y - start_y)
    
    return ind1, ind2

# Configuração do DEAP atualizada
def setup_deap(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, crossover_rate, mutation_rate, initial_positions, previous_best_solution=None, jammer_position=None, epoch_total_length=300):
    toolbox = base.Toolbox()
    
    # Registrar funções
    toolbox.register("individual", create_individual, 
                     num_uavs=num_uavs, 
                     num_timeslots=num_timeslots, 
                     timeslot_length=timeslot_length, 
                     epoch=epoch, 
                     min_y=min_y, 
                     max_y=max_y,
                     epoch_total_length=epoch_total_length,  # ← ADICIONAR
                     previous_best_solution=previous_best_solution,
                     initial_positions=initial_positions,
                     jammer_position=jammer_position)  # Passar a posição do jammer aqui
    
    # O restante do código permanece o mesmo...

    
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate_individual, num_uavs=num_uavs, num_timeslots=num_timeslots, jammer_position=jammer_position)
    
    # Usar partial para fixar o argumento num_uavs na função custom_crossover
    toolbox.register("mate", partial(custom_crossover, num_uavs=num_uavs))
    
    min_x = epoch * num_timeslots * timeslot_length
    max_x = (epoch * num_timeslots + num_timeslots) * timeslot_length
    
    toolbox.register("mutate", mutate_individual, 
                 min_x=min_x, 
                 max_x=max_x, 
                 min_y=min_y, 
                 max_y=max_y, 
                 mutation_rate=mutation_rate,
                 num_uavs=num_uavs,
                 num_timeslots=num_timeslots,
                 epoch=epoch,
                 timeslot_length=timeslot_length,
                 epoch_total_length=epoch_total_length)  # ← ADICIONAR
    
    toolbox.register("select", tools.selTournament, tournsize=3)  # Seleção por torneio
    
    return toolbox

def genetic_algorithm(num_uavs, num_timeslots, timeslot_length, population_size, generations, crossover_rate, mutation_rate, epoch, initial_positions, previous_best_solution=None, jammer_position=None, min_y=0.0, max_y=5.0, epoch_total_length=300):
    # Configurar o DEAP
    toolbox = setup_deap(
        num_uavs=num_uavs,
        num_timeslots=num_timeslots,
        timeslot_length=timeslot_length,
        epoch=epoch,
        min_y=min_y,
        max_y=max_y,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        initial_positions=initial_positions,
        previous_best_solution=previous_best_solution,
        jammer_position=jammer_position,
        epoch_total_length=epoch_total_length  # ← ADICIONAR
    )

    
    # Criar população inicial
    population = toolbox.population(n=population_size)
    
    # Avaliar a população inicial
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
    
    # Configurar estatísticas para impressão
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Listas para armazenar os dados de cada geração
    gen_list = []
    avg_list = []
    std_list = []
    min_list = []
    max_list = []
    
    # Executar o algoritmo genético com eaSimple
    for gen in range(generations):
        # Avançar uma geração
        algorithms.eaSimple(
            population, 
            toolbox, 
            cxpb=crossover_rate,  # Probabilidade de cruzamento
            mutpb=mutation_rate,  # Probabilidade de mutação
            ngen=1,               # Apenas uma geração por iteração
            stats=stats,          # Estatísticas para impressão
            verbose=False         # Desativar impressão da tabela para cada geração
        )
        
        # Coletar os dados da geração atual
        record = stats.compile(population)
        gen_list.append(gen)
        avg_list.append(record["avg"])
        std_list.append(record["std"])
        min_list.append(record["min"])
        max_list.append(record["max"])
        
        # Escrever os valores no arquivo
        #write_fitness_values(epoch, gen, record["avg"], record["max"], record["min"], record["std"])
    
    # Retornar o melhor indivíduo
    best_individual = tools.selBest(population, k=1)[0]
    return best_individual

# Função para gerar posições iniciais dos UAVs
def generate_initial_positions(num_uavs, min_y, max_y, timeslot_length, manual=False, manual_positions=None):
    if manual and manual_positions is not None:
        return manual_positions  # Usa as posições fornecidas

    positions = []
    max_attempts = 1000  # para evitar loops infinitos

    for _ in range(num_uavs):
        attempts = 0
        while True:
            y = random.uniform(min_y, max_y)
            x = random.uniform(0, timeslot_length)  # entre -timeslot_length e 0
            
            candidate = (x, y)
            
            # Verifica se está longe o suficiente das outras posições já geradas
            if all(np.linalg.norm(np.array(candidate) - np.array(pos)) >= MINDIST for pos in positions):
                positions.append(candidate)
                break
            
            attempts += 1
            if attempts >= max_attempts:
                # Se não conseguir, aceita a posição mesmo assim para evitar bloqueio
                positions.append(candidate)
                break
                
    return positions

# def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
#     with open(filename, "a") as file:
#         # Escrever apenas a melhor solução (que já inclui tudo)
#         individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
#         file.write(f"{individual_str}\n \n")

def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            # Processar o conteúdo existente se necessário
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        # Se o arquivo não existir, não há conteúdo para ler
        print("O arquivo não existe. Criando um novo arquivo.")

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")


def write_fitness_values(epoch, gen, avg, max_val, min_val, std, filename="fitness_values.txt"):
    
    with open(filename, "a") as file:
        if gen == 0:  # Escrever o cabeçalho no início de cada epoch
            file.write(f"=== Epoch {epoch + 1} ===\n")
            file.write("gen\tavg\tmax\tmin\tstd\n")
        file.write(f"{gen}\t{avg:.4f}\t{max_val:.4f}\t{min_val:.4f}\t{std:.4f}\n")

# Função principal atualizada
def simulate_uavs_with_ga(num_epochs, num_timeslots, timeslot_length, num_uavs, population_size=50, generations=50, crossover_rate=0.9, mutation_rate=0.3, manual_initial_positions=None, jammer_position=None, min_y=0.0, max_y=10.0, epoch_total_length=300):
    # Gerar posições iniciais dos UAVs
    initial_positions = generate_initial_positions(
        num_uavs, min_y, max_y, timeslot_length,
        manual=manual_initial_positions is not None,
        manual_positions=manual_initial_positions
    )

    previous_best_solution = None
    
    for epoch in range(num_epochs):
        # Executar o algoritmo genético
        best_solution = genetic_algorithm(
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            timeslot_length=timeslot_length,
            population_size=population_size,
            generations=generations,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            epoch=epoch,
            initial_positions=initial_positions,
            previous_best_solution=previous_best_solution,
            jammer_position=jammer_position,  # Passar a posição do jammer aqui
            min_y=min_y,
            max_y=max_y,
            epoch_total_length=epoch_total_length  # ← ADICIONAR
        )

        # O restante do código permanece o mesmo...


        # Salvar a melhor solução em um arquivo de texto
        save_best_solution_to_file(best_solution, initial_positions)
        
        all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness = print_communication_values_per_timeslot(best_solution, num_uavs, num_timeslots, jammer_position)

        # Guardar a melhor solução para a próxima epoch


        # # Calcular e imprimir a coerência dos valores
        # best_fitness = best_solution.fitness.values[0]
        # recalculated_fitness = evaluate_individual(best_solution, num_uavs, num_timeslots, jammer_position)[0]
        # # Ler o último valor máximo do fitness_values.txt
        # with open("fitness_values.txt", "r") as file:
        #     lines = file.readlines()
        #     last_max_fitness = float(lines[-1].split("\t")[2])  # Último max na última linha
        # # Comparar os valores
        # print(f"Fitness da melhor solução: {best_fitness:.4f}")
        # print(f"Fitness recalculado: {recalculated_fitness:.4f}")
        # print(f"Último max no arquivo: {last_max_fitness:.4f}")
        # print(f"São iguais? {'✅' if abs(best_fitness - last_max_fitness) < 1 else '❌'}")
        # # Guardar a melhor solução para a próxima epoch
        # previous_best_solution = best_solution



        previous_best_solution = best_solution

        return best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity


# correr

In [4]:
# Posições que respeitam MINDIST = 5 metros
posicoes_manuais1 = [
    (20.0, 20.0),   # UAV 1
    (20.0, 40.0),   # UAV 2 (60m de distância do UAV1)
    (40.0, 20.0),   # UAV 3 (60m de distância do UAV1)
    (40.0, 40.0)    # UAV 4 (60m de distância dos outros)
]

# Executar simulação
best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity = simulate_uavs_with_ga(
    num_epochs=1, 
    num_timeslots=6, 
    timeslot_length=60, 
    num_uavs=NUM_UAVS,
    population_size=50,
    generations=50,
    manual_initial_positions=posicoes_manuais1,
    jammer_position=np.array([0,500]),
    min_y=0.0,
    max_y=60.0,
    epoch_total_length=120
)

print("Posições iniciais usadas:", initial_positions)
print("Melhor solução encontrada:", best_solution)
print("Posição do jammer:", jammer_position)

Conteúdo existente no arquivo:

Posições iniciais usadas: [(20.0, 20.0), (20.0, 40.0), (40.0, 20.0), (40.0, 40.0)]
Melhor solução encontrada: [20.0, 20.0, 20.0, 40.0, 40.0, 20.0, 40.0, 40.0, 37.4898051986567, 24.17491819856923, 38.54617125787284, 34.54217232054634, 48.050339297297825, 24.314427456431712, 48.768440444678085, 34.468080912611065, 54.9796103973134, 28.349836397138464, 57.09234251574569, 29.084344641092684, 56.10067859459565, 28.628854912863424, 57.53688088935617, 28.936161825222126, 72.4694155959701, 32.52475459570769, 75.63851377361854, 23.626516961639027, 64.15101789189347, 32.94328236929513, 66.30532133403426, 23.40424273783319, 89.9592207946268, 36.69967279427693, 94.18468503149138, 18.16868928218537, 72.2013571891913, 37.25770982572685, 75.07376177871234, 17.87232365044425, 107.4490259932835, 40.87459099284616, 112.73085628936423, 12.710861602731711, 80.25169648648912, 41.57213728215856, 83.84220222339043, 12.340404563055316]
Posição do jammer: [  0 500]


In [5]:
posicoes_manuais1 = [
    (20.0, 20.0),   # UAV 1
    (20.0, 40.0),   # UAV 2 (60m de distância do UAV1)
    (40.0, 20.0),   # UAV 3 (60m de distância do UAV1)
    (40.0, 40.0)    # UAV 4 (60m de distância dos outros)
]

# Lista de posições do jammer
jammer_positions = [
    np.array([0, 500]),
    np.array([200, 500]),
    np.array([400, 500]),
    np.array([600, 500]),
    np.array([800, 500]),
    np.array([1000, 500]),
    np.array([0, 30]),
    np.array([1000, 30]),
]

# Lista de configurações do GA (pop_size, generations)
ga_configs = [
    (50, 50),
    (100, 100),
    (200, 200),
    (500, 500),

]

# Parâmetros fixos
NUM_EPOCHS = 1
NUM_TIMESLOTS = 6
TIMESLOT_LENGTH = 60
EPOCH_TOTAL_LENGTH = 120

# Guardar os resultados
resultados = []

for jammer_pos in jammer_positions:
    for pop_size, generations in ga_configs:
        best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity = simulate_uavs_with_ga(
            num_epochs=NUM_EPOCHS, 
            num_timeslots=NUM_TIMESLOTS, 
            timeslot_length=TIMESLOT_LENGTH, 
            num_uavs=NUM_UAVS,
            population_size=pop_size,
            generations=generations,
            manual_initial_positions=posicoes_manuais1,
            jammer_position=jammer_pos,
            min_y=0.0,
            max_y=60.0,
            epoch_total_length=EPOCH_TOTAL_LENGTH
        )
        
        resultados.append({
            "jammer_position": jammer_pos.tolist(),
            "population_size": pop_size,
            "generations": generations,
            "best_solution": best_solution,
            "avg_min_capacity": avg_min_capacity
        })


# Resumo final
print("\nResumo das simulações:")
for r in resultados:
    print(r)


O arquivo não existe. Criando um novo arquivo.
Conteúdo existente no arquivo:
20.00 20.00 20.00 40.00 40.00 20.00 40.00 40.00 36.17 23.26 35.31 33.33 47.50 21.10 55.10 35.95 52.35 26.53 50.63 26.67 55.01 22.20 70.21 31.90 68.52 29.79 65.94 20.00 62.51 23.31 85.31 27.85 84.70 33.05 81.25 13.34 70.01 24.41 100.42 23.79 100.87 36.32 96.56 6.67 77.51 25.51 115.52 19.74

Conteúdo existente no arquivo:
20.00 20.00 20.00 40.00 40.00 20.00 40.00 40.00 36.17 23.26 35.31 33.33 47.50 21.10 55.10 35.95 52.35 26.53 50.63 26.67 55.01 22.20 70.21 31.90 68.52 29.79 65.94 20.00 62.51 23.31 85.31 27.85 84.70 33.05 81.25 13.34 70.01 24.41 100.42 23.79 100.87 36.32 96.56 6.67 77.51 25.51 115.52 19.74

20.00 20.00 20.00 40.00 40.00 20.00 40.00 40.00 33.82 24.52 33.60 34.56 53.92 24.37 44.07 34.58 47.65 29.04 47.20 29.11 67.84 28.74 48.15 29.15 61.47 33.56 60.80 23.67 81.75 33.11 52.22 23.73 75.30 38.09 74.40 18.22 95.67 37.49 56.30 18.30 89.12 42.61 88.00 12.78 109.59 41.86 60.37 12.88

Conteúdo existente 

# analisar

In [4]:
def analisar_comunicacoes_detalhado(filename, num_uavs, num_timeslots, jammer_position):
    def parse_solution_file(filename):
        with open(filename, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]
        
        solutions = []
        current_solution = []
        for line in lines:
            if line.startswith('==='):  # Nova época
                if current_solution:
                    solutions.append(current_solution)
                    current_solution = []
            else:
                current_solution.extend(map(float, line.split()))
        if current_solution:
            solutions.append(current_solution)
        return solutions

    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # Processar arquivo de saída
    solucoes = parse_solution_file(filename)
    
    resultados = []
    for idx, solucao in enumerate(solucoes):
        print(f"\nAnálise para Época {idx+1}:")
        
        # 🚨 VERIFICAR COLISÕES PRIMEIRO (IGUAL AO evaluate_individual)
        #tem_colisoes = has_collision(solucao, num_uavs, num_timeslots)
        #print(f"🔍 Verificação de colisões: {'❌ TEM COLISÕES' if tem_colisoes else '✅ SEM COLISÕES'}")
        
        # if tem_colisoes:
        #     print(f"🚨 SOLUÇÃO COM COLISÕES DETECTADA!")
        #     print(f"   Fitness = 0.0 (igual ao evaluate_individual)")
        #     print(f"   Não será feita análise detalhada.")
        #     print("="*50)
        #     return [(0.0, "Solução com colisões")]
        
        # Se não há colisões, continuar com a análise normal
        print(f"✅ Solução válida - prosseguindo com análise detalhada...")
        
        # Lista para armazenar todos os C_média_total e C_min_total do epoch
        todos_c_media = []
        todos_c_min = []
        
        # Recriar a solução para cada timeslot
        for t in range(num_timeslots):
            #print(f"\nTimeslot {t+1}:")
            positions = []
            angles = []
            
            for i in range(num_uavs):
                idx_pos = t * num_uavs * 2 + i * 2  # MUDANÇA: *2, pois só temos x e y
                x, y = solucao[idx_pos:idx_pos+2]
                positions.append(np.array([x, y]))
                
                # Calcular ângulo dinamicamente
                angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
                angles.append(angle)
            
            # Calcular matriz de comunicação
            comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
            
            # Gerar métricas detalhadas
            metricas = avaliar_grafo(comm_matrix)
            
            # Exibir resultados
            #print("\nMatriz de Comunicação (bps):")
            #print(np.round(comm_matrix, 2))
            
            #print("\nResumo de Capacidades:")
            #print(f"- Capacidades: {metricas['capacidades']}")
            #print(f"- Média: {np.mean(metricas['capacidades']):.2f} bps")
            #print(f"- Mínima: {min(metricas['capacidades']):.2f} bps")
            #print(f"- Máxima: {max(metricas['capacidades']):.2f} bps")
            
            #print("\nCaminhos Críticos:")
            #for (i,j), data in metricas['caminhos_minimos'].items():
                #print(f"UAV {i} → UAV {j}: {data['path']} (Capacidade: {data['capacidade']:.2f} bps)")
            
            #print("\nBottlenecks por UAV:")
            #for uav, cap in metricas['bottlenecks'].items():
                #print(f"UAV {uav}: {cap:.2f} bps")
            
            #print(f"\nGrafo é fortemente conexo? {'Sim' if metricas['conectividade'] else 'Não'}")
            
            #print("\nLinks Usados:")
            #for u, v in metricas['links_usados']:
                #print(f"{u} ↔ {v}")
            
            #print("\nLinks Não Usados:")
            #for u, v in metricas['links_nao_usados']:
                #print(f"{u} ↔ {v}")
            
            # Cálculo do fitness usando apenas links usados
            capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
            if capacidades_usadas:
                C_media_total = np.mean(capacidades_usadas)
                C_min_total = min(capacidades_usadas)
            else:
                C_media_total = 0
                C_min_total = 0
            
            # Adicionar às listas do epoch
            todos_c_media.append(C_media_total)
            todos_c_min.append(C_min_total)
            
            #print(f"\nCálculo do Fitness para Timeslot {t+1}:")
            #print(f"Capacidades Usadas: {capacidades_usadas}")
            #print(f"C_média_total = {C_media_total:.2f} bps")
            #print(f"C_min_total = {C_min_total:.2f} bps")
            
            resultados.append({
                'epoch': idx+1,
                'timeslot': t+1,
                'metricas': metricas,
                'C_media_total': C_media_total,
                'C_min_total': C_min_total
            })
        
        # ✅ CORREÇÃO: Calcular fitness de cada timeslot e depois fazer a média (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness_por_timeslot = []
        
        for i in range(len(todos_c_media)):
            C_media = todos_c_media[i]
            C_min = todos_c_min[i]
            fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
            fitness_por_timeslot.append(fitness_timeslot)
        
        # Fitness médio do epoch (IGUAL AO evaluate_individual)
        fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
        
        # Calcular também as médias para informação
        media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
        media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
        
        print(f"\n" + "="*50)
        print(f"RESUMO DO EPOCH {idx+1}:")
        print(f"Todos os C_média_total: {[f'{x:.2f}' for x in todos_c_media]}")
        print(f"Todos os C_min_total: {[f'{x:.2f}' for x in todos_c_min]}")
        print(f"Fitness por timeslot: {[f'{x:.4f}' for x in fitness_por_timeslot]}")
        print(f"Média dos C_média_total: {media_c_media_epoch:.2f} bps")
        print(f"Média dos C_min_total: {media_c_min_epoch:.2f} bps")
        print(f"🎯 Fitness do Epoch (CORRETO) = {fitness_epoch:.4f}")
        print(f"✅ Agora deve bater com o valor do gráfico!")
        print("="*50)
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch


In [11]:
resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch = analisar_comunicacoes_detalhado(
    filename="best_solution_epoch.txt",
    num_uavs=4,  # ou o número correto de UAVs que você está usando
    num_timeslots=6,  # Confirmando que são 6 timeslots
    jammer_position=np.array([0.0,500.0])  # Posição do jammer
)

print(media_c_media_epoch)




Análise para Época 1:
✅ Solução válida - prosseguindo com análise detalhada...

RESUMO DO EPOCH 1:
Todos os C_média_total: ['11090.53', '18924.09', '331068.77', '18114.23', '10178.95', '6951.23']
Todos os C_min_total: ['1219.27', '2428.20', '242661.74', '3583.80', '2274.70', '1785.62']
Fitness por timeslot: ['13522349.0985', '45951410.9729', '80337722566.6863', '64917751.1183', '23154057.2623', '12412259.6850']
Média dos C_média_total: 66054.63 bps
Média dos C_min_total: 42325.55 bps
🎯 Fitness do Epoch (CORRETO) = 13416280065.8039
✅ Agora deve bater com o valor do gráfico!
66054.63190064249


In [9]:
import os
import numpy as np
import tempfile
import traceback

# === Configurações ===
INPUT_FILE = "best_solution_epoch.txt"
NUM_UAVS = 4
NUM_TIMESLOTS = 6

jammer_positions = [
    np.array([0, 500]),
    # np.array([200, 500]),
    # np.array([400, 500]),
    # np.array([600, 500]),
    # np.array([800, 500]),
    # np.array([1000, 500]),
    # np.array([0, 30]),
    # np.array([1000, 30]),
]
# =====================

def parse_non_empty_lines(filename):
    with open(filename, "r", encoding="utf-8") as f:
        return [line.rstrip() for line in f if line.strip()]

def main():
    if not os.path.exists(INPUT_FILE):
        print(f"Ficheiro de entrada '{INPUT_FILE}' não encontrado.")
        return

    lines = parse_non_empty_lines(INPUT_FILE)
    total = len(lines)
    print(f"Linhas encontradas: {total}")

    for i, line in enumerate(lines):
        block_idx = i // 4
        jammer_pos = jammer_positions[block_idx % len(jammer_positions)]

        # 👇 imprimir linha antes da análise
        print(f"\n=== Linha {i+1}/{total} (bloco {block_idx}, jammer={jammer_pos.tolist()}) ===")
        print(line)

        tmp_name = None
        try:
            # criar ficheiro temporário com apenas esta linha
            with tempfile.NamedTemporaryFile(mode="w", delete=False, encoding="utf-8", suffix=".txt") as tmp:
                tmp_name = tmp.name
                tmp.write(line + "\n")

            # chamar a função como antes, passando o ficheiro
            resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch = analisar_comunicacoes_detalhado(
                filename=tmp_name,
                num_uavs=NUM_UAVS,
                num_timeslots=NUM_TIMESLOTS,
                jammer_position=jammer_pos
            )

            print(f"→ Fitness: {fitness_epoch:.4f}, Média C_media: {media_c_media_epoch:.2f}, Média C_min: {media_c_min_epoch:.2f}")

        except Exception as e:
            print(f"ERRO na linha {i} ({e})")
            traceback.print_exc()

        finally:
            # apagar ficheiro temporário
            if tmp_name and os.path.exists(tmp_name):
                os.remove(tmp_name)

if __name__ == "__main__":
    main()


Linhas encontradas: 1

=== Linha 1/1 (bloco 0, jammer=[0, 500]) ===
20.00 20.00 20.00 40.00 40.00 20.00 40.00 40.00 37.57 26.81 37.40 36.81 47.47 26.83 47.40 36.76 55.14 33.61 54.79 33.62 54.93 33.66 54.81 33.53 72.71 40.42 72.19 30.44 62.40 40.48 62.21 30.29 90.28 47.23 89.59 27.25 69.87 47.31 69.62 27.06 107.85 54.03 106.98 24.06 77.33 54.14 77.02 23.82

Análise para Época 1:
✅ Solução válida - prosseguindo com análise detalhada...

RESUMO DO EPOCH 1:
Todos os C_média_total: ['11090.53', '18924.09', '331068.77', '18114.23', '10178.95', '6951.23']
Todos os C_min_total: ['1219.27', '2428.20', '242661.74', '3583.80', '2274.70', '1785.62']
Fitness por timeslot: ['13522349.0985', '45951410.9729', '80337722566.6863', '64917751.1183', '23154057.2623', '12412259.6850']
Média dos C_média_total: 66054.63 bps
Média dos C_min_total: 42325.55 bps
🎯 Fitness do Epoch (CORRETO) = 13416280065.8039
✅ Agora deve bater com o valor do gráfico!
→ Fitness: 13416280065.8039, Média C_media: 66054.63, Média C